# mb3d in a notebook

3D fractals of Mandelbulb3D from Python and JavaScript ([mb3d-bindings](https://github.com/thr32423/mb3d-bindings)), calculated by the engine of [mb3d-rs](https://github.com/thr32423/mb3d-rs) inside this notebook's own process. EXPERIMENTAL.

Run the cells from the top. The first one installs the two packages when they are missing; each holds the engine with its formulas and maps, so nothing else is needed.

Without a graphics card (as online) the pictures are calculated on the processor: the same pictures, slower.

In [ ]:
import os, platform, subprocess, sys
from IPython.display import Image, display

VERSION = "0.2.0"
RELEASES = "https://github.com/thr32423/mb3d-bindings/releases"
SYSTEM = {"linux": "linux-x64", "win32": "windows-x64", "darwin": "macos-arm64" if platform.machine() == "arm64" else "macos-x64"}[sys.platform]

# the Python package
try:
    import mb3d
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mb3d", "--find-links", f"{RELEASES}/expanded_assets/v{VERSION}"], check=True)
    import mb3d

# the package for Node.js, into this folder's node_modules
if not os.path.exists(os.path.join("node_modules", "mb3d")):
    subprocess.run(f'npm install --silent --no-save "{RELEASES}/download/v{VERSION}/mb3d-{VERSION}-{SYSTEM}.tgz"', shell=True, check=True)

with mb3d.Engine() as engine:
    print("mb3d", engine.version, "with", len(engine.formulas()), "formulas")

## Python

A formula by name comes with a camera that shows it; `engine.open("file.m3p")` takes a parameter file.

In [ ]:
import mb3d

with mb3d.Engine() as engine:
    scene = engine.preset("Amazing Box")          # or engine.open("file.m3p")
    scene.set(zoom=0.3, iterations=12)
    picture = scene.render(width=640, file="box.png")

print(picture)
Image("box.png")

## JavaScript

The same with the module for Node.js. The cell below is a JavaScript program: the notebook hands it to `node`.

In [ ]:
%%script node --input-type=module
import { Engine } from "mb3d";

const engine = await Engine.start();
const scene = await engine.preset("Amazing Box");  // or engine.open("file.m3p")
await scene.set({ zoom: 0.3, iterations: 12, rotation: [25, -40, 0] });
const picture = await scene.render({ width: 640, file: "box-js.png" });
engine.close();

console.log(`${picture.width} x ${picture.height} in ${picture.seconds} s: ${picture.device}`);

In [ ]:
Image("box-js.png")

## Changing the scene

From here on in Python, with one engine kept open. A rendered picture shows itself.

`key = value` settings: the view, the detail, colours, fog, ... `engine.guide()` lists what the keys do.

In [ ]:
engine = mb3d.Engine()
scene = engine.preset("Amazing Box")
scene.set(zoom=0.3, rotation=(25, -40, 0), iterations=14)
scene.render(width=640)

In [ ]:
print(engine.guide()[:1500])

A formula's own values reshape the fractal. `scene.formula_values` shows them; `engine.formulas()` names every formula.

In [ ]:
print(len(engine.formulas()), "formulas")
scene.formula_values

In [ ]:
slot = scene.formula_values[0]["slot"]
scene.set_formula(slot, {"Scale": 2.4})
scene.render(width=640)

## Sliders

Turn, zoom and reshape. The scene keeps what the sliders set.

In [ ]:
from mb3d.notebook import explore

bulb = engine.preset("Integer Power")
explore(bulb)

## Pixels and depth as arrays

For numpy, Pillow, matplotlib: every picture comes with its depth (65535 nearest, 0 background).

In [ ]:
import numpy as np

p = bulb.render(width=480, pixels=True, depth=True)
image = np.frombuffer(p.rgb, np.uint8).reshape(p.height, p.width, 3)
depth = np.frombuffer(p.depth, "<u2").reshape(p.height, p.width)
image.shape, depth.shape, float((depth > 0).mean())

## A sweep

One formula value in eight steps.

In [ ]:
sweep = engine.preset("Amazing Box")
slot = sweep.formula_values[0]["slot"]
for scale in np.linspace(1.6, 3.0, 8):
    sweep.set_formula(slot, {"Scale": float(scale)})
    display(Image(sweep.render(width=240).png))

## Your own parameter files

`engine.open("file.m3p")` takes Mandelbulb3D's `.m3p` and `.m3i` files and text parameters (online: upload the file into the notebook's files first); `scene.save("new.m3p")` writes one.

In [ ]:
scene.save("box.m3p")
again = engine.open("box.m3p")
again.render(width=320)

In [ ]:
engine.close()